# 02 — Transformer Encoder tự xây (Keras)

Nội dung: BiLSTM (mốc để so sánh) → Transformer tự xây → **thí nghiệm bỏ positional encoding** (bước 5) → **khảo sát số head** (bước 6) → **heatmap attention** (bước 10).

> Cần GPU để chạy nhanh (CPU vẫn chạy được nhưng chậm). Có thể đặt `MAX_TRAIN` nhỏ để thử.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

MAX_TRAIN = None   # None = dùng đủ 120.000 mẫu train. Đặt vd. 5000 để chạy thử nhanh.

from data import load_ag_news
import keras_pipeline as kp
import attention_viz as av

print('Thiết bị:', kp.device())
data = load_ag_news(MAX_TRAIN)

## Khối Transformer tự xây
Ba thành phần bắt buộc: ① Positional Encoding · ② Multi-Head Attention · ③ Feed-Forward + Residual + LayerNorm.

In [ ]:
import inspect
from transformer_block import KhoiTransformer, PositionalEncoding

print(inspect.getsource(PositionalEncoding))
print(inspect.getsource(KhoiTransformer))

## BiLSTM — mốc kiến trúc tuần tự (TT-27) để so sánh

In [ ]:
kq_lstm = kp.run_lstm(data)

## Bước 4 — Transformer tự xây: Embedding + Positional Encoding + 2 khối Transformer + GlobalAvgPool + Dense
Padding được che (mask) trong attention và trong bước pooling.

In [ ]:
kq_tf = kp.run_scratch(data)

## Bước 5 — Thí nghiệm: bỏ positional encoding
Hai model giống hệt nhau, chỉ khác có/không có positional encoding. Thêm phép thử **đảo ngẫu nhiên thứ tự từ** trong câu test: model không có positional encoding là *bag-of-words* nên kết quả gần như không đổi.

In [ ]:
df_pos = kp.run_ablation_pos(data)
df_pos

## Bước 6 — Khảo sát số head: 1 / 2 / 4 / 8
`d_model` cố định nên tổng số tham số gần như không đổi; chỉ thay đổi cách chia không gian chú ý.

In [ ]:
df_heads = kp.run_heads(data)

## Bước 10 — Trực quan hoá attention
Mỗi hàng là một tin (3 chuyên mục khác nhau); mỗi cột là một khối Transformer. Ô (hàng *i*, cột *j*) càng sáng nghĩa là từ *i* càng chú ý tới từ *j*.

In [ ]:
mau = av.ve_attention_scratch(data)

Mỗi head là một "góc nhìn" riêng — các head của cùng một khối thường chú ý những mẫu khác nhau:

In [ ]:
av.ve_cac_head(data)